# AWS Cleanup

Terminate EC2 instances and release Elastic IPs prefixed with `FILESYSTEM_NAME`.
Also cleans up the local SSH config.

## Supported Deployments

| Deployment | `FILESYSTEM_NAME` | `SSH_CONFIG_MARKER` | State file |
|---|---|---|---|
| GPFS | `fs1` | `GPFS` | `gpfs-state.json` |
| Lustre | `fs0` | `Lustre` | `lustre-state.json` |
| Dev machine | `flink-dev` | `dev-machine` | `dev-machine-state.json` |

In [ ]:
from __future__ import annotations

import sys
import time

import boto3

sys.path.insert(0, '.')
import logging

from botocore.exceptions import ClientError
from deploy_common import load_deploy_config
from deploy_common import setup_logger
from deploy_common import update_ssh_config

logger, _ = setup_logger('cleanup', logging.INFO)
print('\u2713 Ready')

In [ ]:
# ⚠️ Set DRY_RUN = False only when you are ready to delete resources.
DRY_RUN = False

# Which deployment to clean up: 'gpfs', 'lustre', or 'ingest-dev'
DEPLOYMENT = 'lustre'

# --- Load config from deploy-config.yaml ---
config = load_deploy_config(DEPLOYMENT)
AWS_REGION = config['aws_region']
# Unlike the deploy notebooks (which only need security_group — the SG
# implicitly determines the VPC), cleanup needs vpc_id explicitly to
# *discover* resources by VPC filter.
VPC_ID = config['vpc_id']

_DEPLOYMENT_MAP = {
    'gpfs': ('fs1', 'GPFS', 'gpfs-state.json'),
    'lustre': ('fs0', 'Lustre', 'lustre-state.json'),
    'ingest-dev': ('flink-dev', 'dev-machine', 'dev-machine-state.json'),
}
FILESYSTEM_NAME, SSH_CONFIG_MARKER, STATE_FILE = _DEPLOYMENT_MAP[DEPLOYMENT]

ec2 = boto3.client('ec2', region_name=AWS_REGION)
print(
    f'Region: {AWS_REGION}, VPC: {VPC_ID},'
    f' Prefix: {FILESYSTEM_NAME}, DRY_RUN: {DRY_RUN}',
)

In [ ]:
# Discover instances
response = ec2.describe_instances(
    Filters=[
        {'Name': 'vpc-id', 'Values': [VPC_ID]},
        {'Name': 'tag:Name', 'Values': [f'{FILESYSTEM_NAME}-*']},
        {
            'Name': 'instance-state-name',
            'Values': ['running', 'stopped', 'stopping', 'pending'],
        },
    ],
)
instances = [i for r in response['Reservations'] for i in r['Instances']]
instance_ids = [i['InstanceId'] for i in instances]

print(f'Found {len(instance_ids)} instances:')
for i in instances:
    name = next(
        (t['Value'] for t in i.get('Tags', []) if t['Key'] == 'Name'),
        'N/A',
    )
    print(f'  {i["InstanceId"]} ({name}) - {i["State"]["Name"]}')

In [ ]:
# Discover Elastic IPs associated with VPC instances
# or matching the filesystem name
vpc_enis = {
    eni['NetworkInterfaceId']
    for eni in ec2.describe_network_interfaces(
        Filters=[{'Name': 'vpc-id', 'Values': [VPC_ID]}],
    )['NetworkInterfaces']
}

eips = []
for eip in ec2.describe_addresses()['Addresses']:
    name = next(
        (t['Value'] for t in eip.get('Tags', []) if t['Key'] == 'Name'),
        '',
    )
    if not name.startswith(FILESYSTEM_NAME):
        continue
    if ('InstanceId' in eip and eip['InstanceId'] in instance_ids) or (
        'NetworkInterfaceId' in eip and eip['NetworkInterfaceId'] in vpc_enis
    ):
        eips.append(eip)
    elif 'InstanceId' not in eip and 'NetworkInterfaceId' not in eip:
        eips.append(eip)  # Unassociated EIP with matching name

print(f'Found {len(eips)} Elastic IPs:')
for eip in eips:
    name = next(
        (t['Value'] for t in eip.get('Tags', []) if t['Key'] == 'Name'),
        'N/A',
    )
    print(f'  {eip["AllocationId"]} ({eip.get("PublicIp")}) - {name}')

In [ ]:
# Terminate instances
if DRY_RUN:
    print(f'[DRY RUN] Would terminate {len(instance_ids)} instances')
elif instance_ids:
    print(f'Terminating {len(instance_ids)} instances...')
    ec2.terminate_instances(InstanceIds=instance_ids)
    print('Waiting for termination...')
    ec2.get_waiter('instance_terminated').wait(
        InstanceIds=instance_ids,
        WaiterConfig={'Delay': 10, 'MaxAttempts': 30},
    )
    print('\u2713 All instances terminated')
else:
    print('No instances to terminate')

In [ ]:
# Release Elastic IPs
MAX_RETRIES = 5
RETRY_DELAY = 5  # seconds

if DRY_RUN:
    print(f'[DRY RUN] Would release {len(eips)} Elastic IPs')
elif eips:
    print(f'Releasing {len(eips)} Elastic IPs...')
    for eip in eips:
        ip = eip.get('PublicIp')
        aid = eip['AllocationId']
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                if 'AssociationId' in eip:
                    try:
                        ec2.disassociate_address(
                            AssociationId=eip['AssociationId'],
                        )
                    except ClientError as e:
                        if (
                            e.response['Error']['Code']
                            == 'InvalidAssociationID.NotFound'
                        ):
                            pass  # Already disassociated (instance terminated)
                        else:
                            raise
                    time.sleep(1)
                ec2.release_address(AllocationId=aid)
                print(f'  \u2713 Released {ip} ({aid})')
                break
            except ClientError as e:
                code = e.response['Error']['Code']
                if code == 'InvalidIPAddress.InUse' and attempt < MAX_RETRIES:
                    print(
                        f'  \u26a0 {aid}: still in use, retrying in'
                        f' {RETRY_DELAY}s ({attempt}/{MAX_RETRIES})...',
                    )
                    time.sleep(RETRY_DELAY)
                else:
                    print(f'  \u2717 {aid}: {e}')
                    break
else:
    print('No Elastic IPs to release')

In [ ]:
# Clean up SSH config and cluster state file
import os

from deploy_common import _state_path

state_path = _state_path(STATE_FILE)

if DRY_RUN:
    print('[DRY RUN] Would clean SSH config')
    if os.path.exists(state_path):
        print(f'[DRY RUN] Would delete {STATE_FILE}')
else:
    update_ssh_config([], SSH_CONFIG_MARKER, logger=logger)
    print('✓ SSH config cleaned')
    if os.path.exists(state_path):
        os.remove(state_path)
        print(f'✓ {STATE_FILE} removed')

print('\n' + '=' * 40)
print('CLEANUP COMPLETE' if not DRY_RUN else 'DRY RUN COMPLETE')
print('=' * 40)